# Preprocessing

The exploratory analysis fixed a small set of edits: ages below 18 or above 80 become missing, and four heavy-tailed fields are clipped at a chosen ceiling. This notebook applies those edits to the stratified pool and to the frozen test set, then writes the two tables the models will read. The thresholds were chosen in the EDA and are used here as fixed constants, so the same rule lands on every row of both tables.

### Imports and Data loading


In [ ]:
import pandas as pd
import numpy as np

The EDA saved the stratified cut under data/split. We load the pool and the test set from there. Each table carries eleven columns: the label SeriousDlqin2yrs and the ten application fields. The print is the check that both pieces arrived whole.

In [8]:
pool = pd.read_csv("../data/split/pool.csv")
test = pd.read_csv("../data/split/test.csv")

print(pool.shape, test.shape)

(120000, 11) (30000, 11)


The pool has 120,000 rows and the test set has 30,000. Both have the eleven columns from the split, so the preprocessing starts from the same cut the EDA explored.

### Rules

- Ages under 18 and over 80 are set to missing. On the pool that is one row under 18, and 3,927 rows over 80, which is 3.27 percent of the pool. The oldest recorded age is 109. On the test set there is no age under 18, and 1,009 applicants are over 80. An age of 18 or of 80 stays as it was recorded. The missing ages are empty cells in the saved file, and the column is stored as a decimal field so those empty cells can be written.

- `RevolvingUtilizationOfUnsecuredLines` above 5 is set to 5. Ordinary values sit well below 1, while the raw maximum on the pool is 50,708. The ceiling gathers 199 pool rows, 0.17 percent, and 55 test rows.

- `NumberOfTimes90DaysLate` above 17 is set to 17. By the high teens the genuine tail has already run out, and the raw maximum of 98 is the old ceiling on this field. The new ceiling gathers that old ceiling together with the sparse tail: 214 pool rows and 55 test rows.

- `NumberOfTime30-59DaysPastDueNotWorse` above 6 is stored as 7, which is the replacement tried in the EDA. Afterwards a recorded 7 means more than six episodes, and a recorded 6 still means exactly six. The rule reaches 297 pool rows, 0.25 percent, and 71 test rows, and the old ceiling at 98 is folded into that 7.

- `DebtRatio` above 7,500 is set to 7,500. On the pool the raw maximum is 329,664. The ceiling gathers 340 pool rows, 0.28 percent, and 82 test rows. The larger block of ratios above 10 stays above 10, because 7,500 is itself above 10: that block is 19.18 percent of the pool and 19.53 percent of the test. Those are the ratios the EDA traced to a missing or near-zero monthly income. The ceiling removes the extreme maximum and leaves that block in the table.

Monthly income, number of dependents, open credit lines, real-estate loans, the 60–89 day count, and the label pass through as recorded. Monthly income keeps its missing values, 19.73 percent of the pool, and its tail out to about 3 million. Number of dependents keeps its missing values, 2.61 percent of the pool. The 60–89 day count keeps its recorded ceiling at 98.

In [9]:
def preprocess(df):
    df = df.copy()

    df["age"] = df["age"].apply(lambda x: np.nan if x < 18 else x)
    df["age"] = df["age"].apply(lambda x: np.nan if x > 80 else x)

    df["RevolvingUtilizationOfUnsecuredLines"] = df["RevolvingUtilizationOfUnsecuredLines"].apply(lambda x: 5 if x > 5 else x)

    df["NumberOfTimes90DaysLate"] = df["NumberOfTimes90DaysLate"].apply(lambda x: 17 if x > 17 else x)

    df["NumberOfTime30-59DaysPastDueNotWorse"] = df["NumberOfTime30-59DaysPastDueNotWorse"].apply(lambda x: 7 if x > 6 else x)

    df["DebtRatio"] = df["DebtRatio"].apply(lambda x: 7500 if x > 7500 else x)

    return df

The pool and the test set are each passed through `preprocess`, so both receive the constants above. 

In [10]:
pool = preprocess(pool)
test = preprocess(test)

print(pool.shape, test.shape)

(120000, 11) (30000, 11)


The shapes are the ones we loaded, 120,000 by 11 and 30,000 by 11. Each rule rewrites a value inside an existing cell. The row count and the label stay as they were.

### Modeling tables

The preprocessed pool and test set are written to `data/pool.csv` and `data/test.csv`. 

In [11]:
pool.to_csv("../data/pool.csv", index=False)
test.to_csv("../data/test.csv", index=False)